In [2]:
# filter_single_person.py
from pycocotools.coco import COCO
import shutil
from pathlib import Path

coco = COCO('annotations_trainval2017/annotations/instances_val2017.json')
person_cat_id = coco.getCatIds(catNms=['person'])[0]

output_dir = Path('val2017/val2017_single_person')
output_dir.mkdir(exist_ok=True)

for img_id in coco.getImgIds():
    ann_ids = coco.getAnnIds(imgIds=img_id, catIds=person_cat_id)
    if len(ann_ids) == 1:  # Exactly one person
        img_info = coco.loadImgs(img_id)[0]
        src = Path('val2017/val2017') / img_info['file_name']
        dst = output_dir / img_info['file_name']
        shutil.copy(src, dst)

print(f"Copied {len(list(output_dir.glob('*.jpg')))} single-person images")

loading annotations into memory...
Done (t=0.47s)
creating index...
index created!
Copied 1045 single-person images


In [ ]:
import os
import urllib.request
import zipfile
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
COCO_ROOT = Path("../Coco2017Test")
URLS = {
    "val2017": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}

# ------------------------------------------------------------
# UTILS
# ------------------------------------------------------------
def download(url, dst):
    if dst.exists():
        print(f"[SKIP] {dst.name} already exists")
        return
    print(f"[DOWNLOAD] {url}")
    urllib.request.urlretrieve(url, dst)
    print(f"[DONE] {dst.name}")

def unzip(zip_path, extract_to):
    print(f"[EXTRACT] {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)
    print(f"[DONE] Extracted to {extract_to}")

COCO_ROOT.mkdir(parents=True, exist_ok=True)

# Download archives
val_zip = COCO_ROOT / "val2017.zip"
ann_zip = COCO_ROOT / "annotations_trainval2017.zip"

download(URLS["val2017"], val_zip)
download(URLS["annotations"], ann_zip)

# Extract
unzip(val_zip, COCO_ROOT)
unzip(ann_zip, COCO_ROOT)

print("\nCOCO 2017 download complete.")

[DOWNLOAD] http://images.cocodataset.org/zips/val2017.zip


In [ ]:
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
COCO_ROOT = Path("../Coco2017Test")
URLS = {
    "val2017": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}

CHUNK_SIZE = 1024 * 1024  # 1 MB

# ------------------------------------------------------------
# DOWNLOAD WITH PROGRESS
# ------------------------------------------------------------
def download_with_progress(url, dst):
    if dst.exists():
        print(f"[SKIP] {dst.name} already exists")
        return

    response = requests.get(url, stream=True)
    response.raise_for_status()

    total = int(response.headers.get("content-length", 0))

    with open(dst, "wb") as f, tqdm(
        total=total,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=dst.name,
    ) as pbar:
        for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

# ------------------------------------------------------------
# UNZIP
# ------------------------------------------------------------
def unzip(zip_path, extract_to):
    print(f"[EXTRACT] {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)
    print(f"[DONE] Extracted to {extract_to}")

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():
    COCO_ROOT.mkdir(parents=True, exist_ok=True)

    val_zip = COCO_ROOT / "val2017.zip"
    ann_zip = COCO_ROOT / "annotations_trainval2017.zip"

    download_with_progress(URLS["val2017"], val_zip)
    download_with_progress(URLS["annotations"], ann_zip)

    unzip(val_zip, COCO_ROOT)
    unzip(ann_zip, COCO_ROOT)

    print("\nCOCO 2017 download complete.")

if __name__ == "__main__":
    main()
